# Salarios, inflación e IRPF en España · Preparación de los datos
**08MBID · Visualización de Datos · Trabajo Práctico**

Este cuaderno explica, paso a paso, cómo pasamos de las tablas originales (INE y Agencia Tributaria) a las **tablas de trabajo** que se usan en el análisis y en las visualizaciones del trabajo. Todo el proceso se puede repetir: el cuaderno descarga el repositorio, lee `data/raw` y escribe en `data/processed`.

| Paso | Qué hacemos | Tabla que genera |
|---|---|---|
| 1 | Salarios (INE, EAES): selección y limpieza | `salarios_ccaa.csv` |
| 2 | IPC (INE): media anual por comunidad | `ipc_anual_ccaa.csv` |
| 3 | Salarios frente a precios a nivel nacional | `salario_vs_ipc.csv` |
| 4 | Salarios reales por comunidad autónoma | `ccaa_salario_real.csv`, `ccaa_resumen.csv` |
| 5 | Distribución de salarios por múltiplos del SMI | `distribucion_smi.csv` |
| 6 | Tipo medio del IRPF (AEAT, Modelo 100) | `irpf_tipo_medio_modelo100.csv` |
| 7 | Modelo fiscal: Seguridad Social e IRPF de un asalariado | `modelo_fiscal_resultados.csv`, `curva_bruto_neto.csv`, `parametros_fiscales.csv`, `smi_umbral_irpf.csv` |
| 8 | Comprobaciones finales | — |

**Fuentes:** INE tabla 28191 (salarios), 28182 (distribución sobre SMI), 50913 (IPC); AEAT, Anuario estadístico, descarga masiva (Modelo 100). Enlaces completos en el README del repositorio.

## 0. Preparación
Clonamos el repositorio (si no estamos ya dentro de él) y definimos las rutas y una función para leer los CSV del INE: separador `;`, codificación `latin-1`, coma decimal y punto de millares.

In [1]:
import os, subprocess, sys
import pandas as pd, numpy as np

REPO = "https://github.com/sortaria24-max/08MBID_TrabajoPractico.git"
if not os.path.exists("data/raw"):
    if os.path.exists("../data/raw"):          # ejecutado desde la carpeta notebooks/
        os.chdir("..")
    else:                                       # Colab: descargamos el repositorio
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
        os.chdir("08MBID_TrabajoPractico")

RAW, OUT = "data/raw/", "data/processed/"
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, "src")

def leer(nombre):
    return pd.read_csv(RAW + nombre, sep=";", encoding="latin-1", decimal=",", thousands=".")

def guardar(df, nombre):
    df.to_csv(OUT + nombre, index=False, float_format="%.4f")
    print(f"✔ {nombre}: {len(df)} filas, columnas: {list(df.columns)}")

print("Carpeta de trabajo:", os.path.basename(os.getcwd()))

Carpeta de trabajo: wd


## 1. Salarios (INE, Encuesta Anual de Estructura Salarial)
**Selección.** La tabla trae las series de Hombres, Mujeres y Ambos sexos. El análisis se refiere al conjunto de personas asalariadas, sin distinguir por sexo, así que nos quedamos con **«Ambos sexos»**.

**Qué contiene.** Media, mediana, cuartiles y percentiles 10 y 90 del salario bruto anual por comunidad autónoma, de 2008 a 2024 (serie homogénea con la clasificación CNAE-09). La tabla incluye todas las jornadas: el INE no permite separar aquí la jornada parcial. Ceuta y Melilla no aparecen en esta encuesta.

**Limpieza.** Quitamos la columna de sexo, damos nombres cortos a las medidas y pasamos la tabla de formato largo a formato ancho (una fila por comunidad y año).

In [2]:
sal = leer("Salario_anual.csv")
sal.columns = ["ccaa", "sexo", "medida", "anio", "valor"]
print("Filas originales:", len(sal), "| sexos:", sal.sexo.unique().tolist())
print("Medidas:", sal.medida.unique().tolist())
print("Valores no numéricos o vacíos:", sal.valor.isna().sum())

sal = sal[sal.sexo == "Ambos sexos"].drop(columns="sexo")
sal["ccaa"] = sal.ccaa.replace({"Total Nacional": "Nacional"})
nombres = {"Media": "media", "Mediana": "mediana", "Percentil 10": "p10", "Cuartil inferior": "q1",
           "Cuartil superior": "q3", "Percentil 90": "p90"}
salarios = (sal.assign(medida=sal.medida.map(nombres))
               .pivot_table(index=["ccaa", "anio"], columns="medida", values="valor")
               .reset_index())
salarios.columns.name = None
salarios = salarios[["ccaa", "anio", "p10", "q1", "mediana", "media", "q3", "p90"]].sort_values(["ccaa", "anio"])
guardar(salarios, "salarios_ccaa.csv")
salarios[salarios.ccaa == "Nacional"].tail(3)

Filas originales: 5508 | sexos: ['Ambos sexos', 'Mujeres', 'Hombres']
Medidas: ['Media', 'Percentil 10', 'Cuartil inferior', 'Mediana', 'Cuartil superior', 'Percentil 90']
Valores no numéricos o vacíos: 0
✔ salarios_ccaa.csv: 306 filas, columnas: ['ccaa', 'anio', 'p10', 'q1', 'mediana', 'media', 'q3', 'p90']


,ccaa,anio,p10,q1,mediana,media,q3,p90
252,Nacional,2022,10657.37,15913.52,22383.11,26948.87,33561.62,48217.21
253,Nacional,2023,11466.88,16632.97,23349.00,28049.94,34991.64,49836.00
254,Nacional,2024,12018.46,17457.47,24497.17,29540.26,36969.61,52515.27


## 2. IPC (INE, base 2021)
**Selección.** La tabla original trae cuatro tipos de dato (índice, variación mensual, variación anual y variación en lo que va de año) para cada comunidad. Nos quedamos solo con el **índice general**, porque con el índice se calcula cualquier variación acumulada sin acumular errores de redondeo.

**Transformación.** Los salarios son anuales y el IPC es mensual: calculamos la **media anual** del índice (12 meses). Solo damos por válidos los años con los 12 meses. Quitamos Ceuta y Melilla porque no tienen salarios en la encuesta anterior, y el número que antecede al nombre de la comunidad («01 Andalucía»).

In [3]:
ipc = leer("IPC.csv")
ipc.columns = ["ccaa", "grupo", "tipo", "periodo", "valor"]
print("Tipos de dato:", ipc.tipo.unique().tolist())
print("Grupos ECOICOP:", ipc.grupo.unique().tolist())

ipc = ipc[ipc.tipo == "Índice"].copy()
ipc["anio"] = ipc.periodo.str[:4].astype(int)
ipc["ccaa"] = ipc.ccaa.str.replace(r"^\d+\s+", "", regex=True)
ipc = ipc[~ipc.ccaa.isin(["Ceuta", "Melilla"])]

g = ipc.groupby(["ccaa", "anio"]).valor
ipc_anual = pd.DataFrame({"ipc": g.mean(), "meses": g.count()}).reset_index()
print("Años con menos de 12 meses:", ipc_anual[ipc_anual.meses != 12].shape[0])
ipc_anual = ipc_anual[ipc_anual.meses == 12].drop(columns="meses")
guardar(ipc_anual, "ipc_anual_ccaa.csv")
ipc_anual[ipc_anual.ccaa == "Nacional"].head(3)

Tipos de dato: ['Índice', 'Variación mensual', 'Variación anual', 'Variación en lo que va de año']
Grupos ECOICOP: ['Índice general']
Años con menos de 12 meses: 0
✔ ipc_anual_ccaa.csv: 324 filas, columnas: ['ccaa', 'anio', 'ipc']


,ccaa,anio,ipc
252,Nacional,2008,86.398667
253,Nacional,2009,86.150000
254,Nacional,2010,87.700583


## 3. Salarios frente a precios (nivel nacional)
Para comparar dos magnitudes con unidades distintas (euros e índice de precios) calculamos, para cada año, la **variación acumulada en porcentaje respecto a 2008**, el primer año de la serie: `(valor del año / valor de 2008 − 1) × 100`. Así 2008 vale 0 % y el último dato se lee directamente como «ha subido un X %».

In [4]:
nac = salarios[salarios.ccaa == "Nacional"].merge(ipc_anual[ipc_anual.ccaa == "Nacional"], on=["ccaa", "anio"])
nac = nac[(nac.anio >= 2008) & (nac.anio <= 2024)].sort_values("anio").reset_index(drop=True)
for c in ["media", "mediana", "ipc"]:
    nac[c + "_var_pct"] = 100 * (nac[c] / nac.loc[0, c] - 1)
salario_vs_ipc = nac[["anio", "media", "mediana", "ipc", "media_var_pct", "mediana_var_pct", "ipc_var_pct"]]
guardar(salario_vs_ipc, "salario_vs_ipc.csv")
u = salario_vs_ipc.iloc[-1]
print(f"Acumulado 2008-2024 → salario medio {u.media_var_pct:+.1f} % · mediano {u.mediana_var_pct:+.1f} % · IPC {u.ipc_var_pct:+.1f} %")

✔ salario_vs_ipc.csv: 17 filas, columnas: ['anio', 'media', 'mediana', 'ipc', 'media_var_pct', 'mediana_var_pct', 'ipc_var_pct']
Acumulado 2008-2024 → salario medio +35.0 % · mediano +34.3 % · IPC +33.5 %


## 4. Salarios reales por comunidad autónoma
Para cada comunidad dividimos el salario entre **su propio IPC** (no el nacional), porque los precios no han subido igual en todas partes. El salario real en base 100 es `salario100 / IPC100 × 100`: si vale más de 100, el sueldo ha ganado poder de compra desde 2008; si vale menos, lo ha perdido.

In [5]:
c = salarios.merge(ipc_anual, on=["ccaa", "anio"])
c = c[(c.anio >= 2008) & (c.anio <= 2024)].sort_values(["ccaa", "anio"]).copy()
base = c[c.anio == 2008].set_index("ccaa")
for k in ["media", "mediana", "ipc"]:
    c[k + "100"] = 100 * c[k] / c.ccaa.map(base[k])
c["media_real100"] = 100 * c.media100 / c.ipc100
c["mediana_real100"] = 100 * c.mediana100 / c.ipc100
ccaa_salario_real = c[["ccaa", "anio", "media", "mediana", "ipc", "media100", "mediana100", "ipc100",
                       "media_real100", "mediana_real100"]]
guardar(ccaa_salario_real, "ccaa_salario_real.csv")

u = ccaa_salario_real[ccaa_salario_real.anio == 2024].set_index("ccaa")
resumen = pd.DataFrame({
    "salario_medio_2008": base.media, "salario_medio_2024": u.media,
    "var_salario_pct": u.media100 - 100, "var_ipc_pct": u.ipc100 - 100,
    "var_real_pct": u.media_real100 - 100,
}).reset_index().sort_values("var_real_pct", ascending=False)
# Nombre de cada comunidad tal como lo escribe el mapa de Datawrapper (el INE pone, p. ej., "Madrid, Comunidad de")
NOMBRE_MAPA = {"Balears, Illes": "Islas Baleares", "Murcia, Región de": "Región de Murcia",
               "Navarra, Comunidad Foral de": "Comunidad Foral de Navarra", "Madrid, Comunidad de": "Comunidad de Madrid",
               "Comunitat Valenciana": "Comunidad Valenciana", "Rioja, La": "La Rioja",
               "Asturias, Principado de": "Principado de Asturias", "Castilla - La Mancha": "Castilla-La Mancha"}
resumen.insert(1, "nombre_mapa", resumen.ccaa.replace(NOMBRE_MAPA))
resumen.loc[resumen.ccaa == "Nacional", "nombre_mapa"] = None
guardar(resumen, "ccaa_resumen.csv")
resumen.round(1)

✔ ccaa_salario_real.csv: 306 filas, columnas: ['ccaa', 'anio', 'media', 'mediana', 'ipc', 'media100', 'mediana100', 'ipc100', 'media_real100', 'mediana_real100']
✔ ccaa_resumen.csv: 18 filas, columnas: ['ccaa', 'nombre_mapa', 'salario_medio_2008', 'salario_medio_2024', 'var_salario_pct', 'var_ipc_pct', 'var_real_pct']


,ccaa,nombre_mapa,salario_medio_2008,salario_medio_2024,var_salario_pct,var_ipc_pct,var_real_pct
4,Canarias,Canarias,18467.0,25120.4,36.0,27.9,6.3
3,"Balears, Illes",Islas Baleares,20388.1,29075.1,42.6,34.8,5.8
13,"Murcia, Región de",Región de Murcia,18911.8,26349.4,39.3,31.9,5.6
15,"Navarra, Comunidad Foral de",Comunidad Foral de Navarra,23342.7,32605.3,39.7,33.0,5.0
12,"Madrid, Comunidad de",Comunidad de Madrid,25269.6,34410.0,36.2,31.3,3.7
10,Extremadura,Extremadura,18264.9,24979.4,36.8,31.9,3.7
11,Galicia,Galicia,19156.4,26547.3,38.6,34.8,2.8
9,Comunitat Valenciana,Comunidad Valenciana,19793.6,26817.0,35.5,32.7,2.1
16,País Vasco,País Vasco,25547.1,35170.3,37.7,35.6,1.5
14,Nacional,NaN,21883.4,29540.3,35.0,33.5,1.1


## 5. Distribución de los salarios por múltiplos del SMI
La tabla 28182 del INE dice qué porcentaje de trabajadores cobra entre 0 y 1 SMI, entre 1 y 2 SMI, etc. Primero limpiamos la tabla: nos quedamos con «Ambos sexos», quitamos la fila «Total de trabajadores» (siempre 100 %) y comprobamos que los tramos suman 100 %.

*Aviso de lectura:* el SMI sube cada año (en 2019 un 22 %), así que un tramo «de 1 a 2 SMI» no equivale a los mismos euros en todos los años.

In [6]:
d = leer("Distribucion_salarial_tramos.csv")
d.columns = ["jornada", "tramo", "sexo", "anio", "pct"]
print("Sexos:", d.sexo.unique().tolist(), "| jornadas:", d.jornada.unique().tolist())
d = d[(d.sexo == "Ambos sexos") & (d.tramo != "Total de trabajadores")].drop(columns="sexo")
d["jornada"] = d.jornada.replace({"Total": "Todas las jornadas", "Jornada a tiempo completo": "Tiempo completo"})
d["orden"] = d.tramo.str.extract(r"(\d+) SMI").astype(float).fillna(0).astype(int)
d.loc[d.tramo == "Más de 8 SMI", "orden"] = 9
d = d.sort_values(["jornada", "anio", "orden"]).reset_index(drop=True)
suma = d.groupby(["jornada", "anio"]).pct.sum()
print("Suma de tramos (mín/máx):", round(suma.min(), 2), round(suma.max(), 2))
d[(d.jornada == "Todas las jornadas") & (d.anio == 2024)]

Sexos: ['Ambos sexos', 'Mujeres', 'Hombres'] | jornadas: ['Total', 'Jornada a tiempo completo']
Suma de tramos (mín/máx): 99.99 100.01


,jornada,tramo,anio,pct,orden
297,Todas las jornadas,De 0 a 1 SMI,2024,18.32,1
298,Todas las jornadas,De 1 a 2 SMI,2024,48.69,2
299,Todas las jornadas,De 2 a 3 SMI,2024,19.62,3
300,Todas las jornadas,De 3 a 4 SMI,2024,7.72,4
301,Todas las jornadas,De 4 a 5 SMI,2024,2.98,5
302,Todas las jornadas,De 5 a 6 SMI,2024,1.54,6
303,Todas las jornadas,De 6 a 7 SMI,2024,0.81,7
304,Todas las jornadas,De 7 a 8 SMI,2024,0.21,8
305,Todas las jornadas,Más de 8 SMI,2024,0.12,9


**Tabla de trabajo.** Para ver la evolución año a año guardamos una tabla con **una fila por año y una columna por grupo**. Nos quedamos con «Todas las jornadas» y agrupamos los nueve tramos en cinco: de 0 a 1, de 1 a 2, de 2 a 3 y de 3 a 4 SMI, y **más de 4 SMI**, que suma los cinco tramos superiores (en 2024 reúnen solo el 5,7 % de los trabajadores). Cada fila sigue sumando 100 %.

In [7]:
grupos = {1: "smi_0_1", 2: "smi_1_2", 3: "smi_2_3", 4: "smi_3_4"}   # orden 5 a 9 → más de 4 SMI
t = d[d.jornada == "Todas las jornadas"].copy()
t["grupo"] = t.orden.map(grupos).fillna("smi_mas_4")
smi_anual = (t.pivot_table(index="anio", columns="grupo", values="pct", aggfunc="sum")
               [["smi_0_1", "smi_1_2", "smi_2_3", "smi_3_4", "smi_mas_4"]]
               .reset_index())
smi_anual.columns.name = None
assert smi_anual.drop(columns="anio").sum(axis=1).sub(100).abs().max() < 0.1, "Algún año no suma 100 %"
guardar(smi_anual, "distribucion_smi.csv")
smi_anual.round(1)

✔ distribucion_smi.csv: 17 filas, columnas: ['anio', 'smi_0_1', 'smi_1_2', 'smi_2_3', 'smi_3_4', 'smi_mas_4']


,anio,smi_0_1,smi_1_2,smi_2_3,smi_3_4,smi_mas_4
0,2008,8.9,34.4,28.3,13.3,15.1
1,2009,10.0,34.2,27.4,13.1,15.3
2,2010,10.5,33.9,27.4,13.3,14.9
3,2011,11.3,33.2,27.2,13.6,14.8
4,2012,12.2,33.2,26.9,13.2,14.5
5,2013,13.3,33.0,26.1,13.0,14.7
6,2014,13.0,32.1,26.8,13.4,14.7
7,2015,12.6,32.3,26.5,13.4,15.1
8,2016,12.6,33.2,26.3,13.3,14.7
9,2017,13.1,36.9,25.2,12.5,12.4


## 6. Tipo medio del IRPF (AEAT, Modelo 100)
El Anuario estadístico de la AEAT publica los datos agregados de las declaraciones por comunidad, sexo y nacionalidad. Sumamos todas las filas de cada año y calculamos el **tipo medio = cuota íntegra total / base liquidable total** (columnas `M100_25` y `M100_17`; el significado de cada columna está en `AyudaCSV_AnuarioTotal.pdf`).

*Alcance:* el fichero no incluye País Vasco ni Navarra (regímenes forales) ni Ceuta y Melilla. Mezcla todos los declarantes y todas las fuentes de renta, así que **sirve para contrastar la tendencia, no para medir a un asalariado concreto** (eso lo hace el paso 7).

In [8]:
m = pd.read_csv(RAW + "modelo100.csv", sep=";", decimal=",", encoding="latin-1")
cols = {"M100_1": "ingresos_trabajo", "M100_17": "base_liquidable", "M100_25": "cuota_integra",
        "M100_34": "declarantes"}
t = m.groupby("EJER")[list(cols)].sum().rename(columns=cols).reset_index().rename(columns={"EJER": "anio"})
t["tipo_medio_pct"] = 100 * t.cuota_integra / t.base_liquidable
guardar(t, "irpf_tipo_medio_modelo100.csv")
print(f"Tipo medio 2015: {t.loc[t.anio==2015,'tipo_medio_pct'].item():.1f} % → 2024: {t.loc[t.anio==2024,'tipo_medio_pct'].item():.1f} %")
t[["anio", "declarantes", "tipo_medio_pct"]].tail(4)

✔ irpf_tipo_medio_modelo100.csv: 16 filas, columnas: ['anio', 'ingresos_trabajo', 'base_liquidable', 'cuota_integra', 'declarantes', 'tipo_medio_pct']
Tipo medio 2015: 18.9 % → 2024: 21.8 %


,anio,declarantes,tipo_medio_pct
12,2021,24696889,20.567177
13,2022,25455149,20.819134
14,2023,26423407,21.226104
15,2024,26932445,21.799362


## 7. Modelo fiscal: ¿cuánto se lleva Hacienda de un sueldo típico?
Las tablas anteriores dicen cuánto se cobra, pero no cuánto impuesto pagaría una persona con ese sueldo. Lo calculamos con un **modelo simplificado** (código en `src/modelo_fiscal.py`, con 8 pruebas hechas a mano).

**Contribuyente tipo.** Asalariado del régimen general con contrato indefinido, sin otras rentas ni deducciones personales. Solo restamos lo que la ley concede automáticamente a cualquier asalariado: la **cotización a la Seguridad Social del trabajador**, los **2.000 € de gastos generales** y la **reducción por rendimientos del trabajo** (que desaparece al subir el sueldo). Se calcula con el mínimo personal (5.550 €) y con la escala estatal más una autonómica que suponemos igual (la tarifa consolidada es el doble de la estatal). Los regímenes forales no se modelan.

**Escenario «no deflactar».** Comparamos el IRPF real de cada año con el que pagaría la misma persona si la tarifa, los mínimos y los gastos se hubieran actualizado con el IPC desde 2016. Es una estimación con supuestos explícitos, no una cifra oficial.

**Perfiles:** salario medio (persona sola), salario mediano (persona sola) y familia de dos adultos con salario medio y mediano y dos hijos (con tributación individual o conjunta, la más favorable).

In [9]:
import importlib, modelo_fiscal as mf
importlib.reload(mf)
assert mf.pruebas(), "Alguna prueba del modelo fiscal falla"

OK   SS 30.000 € (2024) 1941.0 esperado 1941.0
OK   IRPF soltero 30.000 € (2024) 4928.7 esperado 4928.7
OK   IRPF con SMI 2024 0.0 esperado 0.0
OK   IRPF con SMI 2023 39.4509 esperado 39.45
OK   Escala estatal 60.000 (2025) 8950.75 esperado 8950.75
OK   Reducción 2024 en 17.673,52 2364.34 esperado 2364.34
OK   Reducción 2023 en 14.047,5 6498.0 esperado 6498.0
OK   monotonía del neto (2024)

TODAS LAS PRUEBAS OK


In [10]:
filas = mf.calcular(RAW + "Salario_anual.csv", RAW + "IPC.csv")
res = pd.DataFrame(filas)
guardar(res, "modelo_fiscal_resultados.csv")
res[(res.anio == 2024)][["perfil", "bruto", "ss_trabajador", "irpf_real", "irpf_tarifa_indexada_ipc",
                         "diferencia", "tipo_irpf_real_pct", "neto"]]

✔ modelo_fiscal_resultados.csv: 30 filas, columnas: ['anio', 'perfil', 'bruto', 'ss_trabajador', 'irpf_real', 'irpf_tarifa_indexada_ipc', 'diferencia', 'tipo_irpf_real_pct', 'tipo_irpf_indexado_pct', 'neto', 'modalidad_real', 'modalidad_indexada', 'factor_ipc_vs_2016']


,perfil,bruto,ss_trabajador,irpf_real,irpf_tarifa_indexada_ipc,diferencia,tipo_irpf_real_pct,neto
27,media_individual,29540.26,1911.25,4799.70,3972.19,827.51,16.25,22829.30
28,mediana_individual,24497.17,1584.97,3384.66,2830.34,554.32,13.82,19527.54
29,familia_media_mas_mediana,54037.43,3496.22,7215.36,5603.70,1611.66,13.35,43325.85


In [11]:
# Curva bruto → neto (sueldo bruto, cotizaciones, IRPF y neto): un soltero, de 10.000 a 100.000 €
ipc_nac = mf.cargar_ipc_medio(RAW + "IPC.csv")
filas = []
for y in [2015, 2019, 2024]:
    f = mf.factor(y, ipc_nac)
    for b in range(10000, 100001, 500):
        irpf, ss = mf.irpf_individual(b, y, 0.0, 1.0)
        idx, _ = mf.irpf_individual(b, y, 0.0, f)
        filas.append(dict(anio=y, bruto=b, ss=ss, irpf=irpf, irpf_indexado=idx, neto=b - ss - irpf))
curva = pd.DataFrame(filas)
guardar(curva, "curva_bruto_neto.csv")

# Parámetros usados, año a año (transparencia: cada cifra se puede contrastar con la ley)
par = []
for y in mf.AÑOS:
    u = mf.umbrales_escala(y)
    par.append(dict(anio=y, umbrales_escala_estatal=" / ".join(f"{x:,.0f}" for x in u),
                    tipos_estatal_pct=" / ".join(str(x) for x in mf.TIPOS_ESTATAL[:len(u)]),
                    base_maxima_cotizacion_mensual=mf.BASE_MAX_MENSUAL[y], tipo_ss_trabajador_pct=round(mf.tipo_ss(y), 2),
                    minimo_contribuyente=mf.MINIMO_CONTRIBUYENTE, gastos_generales=mf.GASTOS_OTROS))
guardar(pd.DataFrame(par), "parametros_fiscales.csv")

✔ curva_bruto_neto.csv: 543 filas, columnas: ['anio', 'bruto', 'ss', 'irpf', 'irpf_indexado', 'neto']
✔ parametros_fiscales.csv: 11 filas, columnas: ['anio', 'umbrales_escala_estatal', 'tipos_estatal_pct', 'base_maxima_cotizacion_mensual', 'tipo_ss_trabajador_pct', 'minimo_contribuyente', 'gastos_generales']


**Salario mínimo y umbral del IRPF.** Para cada año buscamos, con el modelo fiscal, el **sueldo bruto a partir del cual un asalariado soltero empieza a pagar IRPF** (de euro en euro) y lo comparamos con el **salario mínimo interprofesional (SMI)** anual, en 14 pagas. Los valores del SMI son los de los reales decretos publicados en el BOE:

| Año | €/mes | Real decreto |
|---|---|---|
| 2015 | 648,60 | RD 1106/2014 |
| 2016 | 655,20 | RD 1171/2015 |
| 2017 | 707,60 | RD 742/2016 |
| 2018 | 735,90 | RD 1077/2017 |
| 2019 | 900,00 | RD 1462/2018 |
| 2020 | 950,00 | RD 231/2020 |
| 2021 | 965,00 | RD 817/2021 (desde el 1 de septiembre; hasta agosto, 950 €) |
| 2022 | 1.000,00 | RD 152/2022 |
| 2023 | 1.080,00 | RD 99/2023 |
| 2024 | 1.134,00 | RD 145/2024 |

In [12]:
SMI_MENSUAL = {2015: 648.60, 2016: 655.20, 2017: 707.60, 2018: 735.90, 2019: 900.00,
               2020: 950.00, 2021: 965.00, 2022: 1000.00, 2023: 1080.00, 2024: 1134.00}   # BOE, 14 pagas

def umbral_irpf(y):
    # Primer sueldo bruto (de euro en euro) con el que el modelo da una cuota de IRPF mayor que cero
    b = 8000
    while mf.irpf_individual(b, y)[0] <= 0:
        b += 1
    return b

smi = pd.DataFrame([dict(anio=y, smi_mensual=m, smi_anual=m * 14, umbral_irpf=umbral_irpf(y),
                         irpf_con_smi=mf.irpf_individual(m * 14, y)[0]) for y, m in SMI_MENSUAL.items()])
guardar(smi, "smi_umbral_irpf.csv")
smi.round(2)

✔ smi_umbral_irpf.csv: 10 filas, columnas: ['anio', 'smi_mensual', 'smi_anual', 'umbral_irpf', 'irpf_con_smi']


,anio,smi_mensual,smi_anual,umbral_irpf,irpf_con_smi
0,2015,648.6,9080.4,12013,0.00
1,2016,655.2,9172.8,12013,0.00
2,2017,707.6,9906.4,12013,0.00
3,2018,735.9,10302.6,12644,0.00
4,2019,900.0,12600.0,14005,0.00
5,2020,950.0,13300.0,14005,0.00
6,2021,965.0,13510.0,14005,0.00
7,2022,1000.0,14000.0,14005,0.00
8,2023,1080.0,15120.0,15017,39.45
9,2024,1134.0,15876.0,15880,0.00


## 8. Comprobaciones finales
Antes de usar las tablas, contrastamos las cifras clave con el dato publicado y comprobamos que no hay huecos.

In [13]:
s = salarios[(salarios.ccaa == "Nacional") & (salarios.anio == 2024)].iloc[0]
assert round(s.media, 2) == 29540.26, "El salario medio de 2024 no coincide con el INE"
u = salario_vs_ipc.iloc[-1]
assert abs(u.ipc_var_pct - 33.5) < 0.1 and abs(u.media_var_pct - 35.0) < 0.1
assert salarios.drop(columns="ccaa").notna().all().all()
assert ccaa_salario_real.groupby("ccaa").anio.count().eq(17).all()
assert res.groupby("perfil").anio.count().eq(10).all()

# El tipo medio del modelo (asalariado típico) y el de la AEAT (todos los declarantes) deben moverse en el mismo sentido
p = res[res.perfil == "mediana_individual"].set_index("anio").tipo_irpf_real_pct
a = t.set_index("anio").tipo_medio_pct
print(f"Tipo de IRPF 2015→2024 · nuestro modelo (mediana): {p[2015]:.1f} % → {p[2024]:.1f} %  |  AEAT (todos los declarantes): {a[2015]:.1f} % → {a[2024]:.1f} %")

for f in sorted(os.listdir(OUT)):
    print(f"{f:36s} {len(pd.read_csv(OUT + f)):5d} filas")
print("\nTodo correcto ✔")

Tipo de IRPF 2015→2024 · nuestro modelo (mediana): 11.4 % → 13.8 %  |  AEAT (todos los declarantes): 18.9 % → 21.8 %


ccaa_resumen.csv                        18 filas
ccaa_salario_real.csv                  306 filas
curva_bruto_neto.csv                   543 filas
distribucion_smi.csv                    17 filas
ipc_anual_ccaa.csv                     324 filas
irpf_tipo_medio_modelo100.csv           16 filas
modelo_fiscal_resultados.csv            30 filas
parametros_fiscales.csv                 11 filas
salario_vs_ipc.csv                      17 filas
salarios_ccaa.csv                      306 filas
smi_umbral_irpf.csv                     10 filas

Todo correcto ✔


## 9. Descargar las tablas (solo en Colab)
En Colab las tablas se crean en una carpeta temporal. Esta celda las comprime en `tablas_processed.zip` y lo descarga, para subirlas a `data/processed` del repositorio. Fuera de Colab no hace nada.

In [14]:
import shutil
try:
    from google.colab import files
    shutil.make_archive("tablas_processed", "zip", OUT)
    files.download("tablas_processed.zip")
except ImportError:
    print("No estamos en Colab: las tablas ya están en", OUT)

No estamos en Colab: las tablas ya están en data/processed/
